# 7.2 性能建模与 Roofline 模型

## 1. 三个基本量

任何算子的性能都由三个量刻画：

| 量 | 定义 | 说明 |
|--|--|--|
| W（计算量） | 算子需要做的乘加次数 | FA 里是 mm1 + mm2 的 MAC 数 |
| Q（访存量） | 算子需要搬运的字节数 | 从哪一级搬到哪一级要分开算 |
| I（计算强度） | I = W / Q | 每搬 1 字节能做多少次计算 |

给定带宽 B（字节/秒），搬运耗时 t = Q / B，实际达到的性能 P = W / t = B·I。若计算资源上限为 Pm（乘加/秒），则：

$$P = \begin{cases} B_m \cdot I, & I \leq P_m / B_m \quad \text{（带宽受限）} \\ P_m, & I > P_m / B_m \quad \text{（计算受限）} \end{cases}$$

对数坐标下这是一条斜率为 1 的直线（ln P = ln I + ln B）加一条水平线，交点在拐点 **I\* = Pm / Bm**——这就是著名的 Roofline（屋顶线）形状。

![Roofline](images/roofline.svg)

## 2. FA 的 W 与 Q：以矩阵乘基本块为单位

对一个 (M, N, K) 的矩阵乘，按基本块 (BaseM, BaseN, BaseK) 分块循环（`M_loops = M/BaseM` 以此类推）：

- 计算量：`W = M_loops · N_loops · K_loops · (BaseM·BaseN·BaseK) = M·N·K`——分块不改变计算量；
- 访存量：每个基本块要搬入 A 块（BaseM×BaseK）和 B 块（BaseN×BaseK）：

```
Q = M·N·K · (1/BaseM + 1/BaseN) · D_size
```

于是计算强度：

$$I = \frac{W}{Q} = \frac{BaseM \cdot BaseN}{(BaseM + BaseN) \cdot D_{size}}$$

**这是全课程最重要的结论之一：计算强度只由基本块大小决定**——块越大，每字节搬运支撑的计算越多，越偏向计算受限。这正是「切块」成为优化手段的理论依据（第 5 章 Tiling 原则一的深化）。

## 3. 混合带宽：MTE2 的真实 B 不是 HBM 带宽

数据并非全部来自 HBM——一部分命中 L2 cache。设 ∂ 为 HBM 访存量占比（未命中率），(1−∂) 为 L2 占比：

$$t = \frac{\partial \cdot Q}{B_{HBM}} + \frac{(1-\partial) \cdot Q}{B_{L2cache}} \quad\Rightarrow\quad B_{mix} = \frac{B_{HBM} \cdot B_{L2cache}}{\partial \cdot B_{L2cache} + (1-\partial) \cdot B_{HBM}}$$

- ∂ = 1（全 miss）：B_mix = B_HBM；∂ = 0（全命中）：B_mix = B_L2——**命中率越高，有效带宽越大**；
- FA 的 KV Cache 复用性差（decode 时每个 K/V 元素只被读一次），∂ 接近 1，这就是 decode 带宽受限的根源；
- 计算 bound 的条件变为 I > Pm / B_mix，其中 ∂ 本身也是 (M, N) 与块大小的函数（`∂ = (M+N)/(M·N) · BaseM·BaseN/(BaseM+BaseN)`）。

## 4. MTE1 / L0 一级同理

AIV 侧把数据从 UB 搬进 Cube 的 L0A/L0B（MTE1），推导完全同构：L0A 访存量 Q_L0A = M·N·K·D_size/BaseN，L0B 访存量 Q_L0B = M·N·K·D_size/BaseM，混合带宽 B_mix = (BaseM+BaseN)/(BaseM/B_L0A + BaseN/B_L0B)，介于两条 L0 带宽之间。**每一级搬运都可以套同一个 Roofline 框架**——只是 W、Q、B 的含义换成该级的量。

## 小测验

1. 写出 Roofline 的分段公式与拐点。I < I* 时算子处于什么状态，该吃什么药？
2. 推导矩阵乘基本块的计算强度公式，说明为什么「块越大 I 越高」。块大小的上限由什么限制？
3. B_mix 公式里 ∂ 是什么？∂ = 1 和 ∂ = 0 分别对应什么场景？FA decode 的 ∂ 为什么接近 1？
4. FA 的时间画像 V1 > C1 ≈ C2 >> V2 说明什么 bound？此时优先优化什么？

In [ ]:
# 动手：给 FA 的 mm 建一个 Roofline 账单（数字为教学用模拟参数）
import numpy as np

# ---- 「硬件」参数（模拟量纲，方法与真实规格表一致） ----
Pm       = 60e12    # Cube 峰值乘加/秒
B_HBM    = 1.2e12   # HBM 字节/秒
B_L2     = 6.0e12   # L2 字节/秒
D_size   = 2        # FP16 每元素 2 字节

def intensity(BaseM, BaseN):
    # 基本块计算强度 I = BaseM*BaseN / ((BaseM+BaseN)*D_size)
    return BaseM * BaseN / (BaseM + BaseN) / D_size

def roofline(BaseM, BaseN, partial=1.0):
    I = intensity(BaseM, BaseN)
    B_mix = B_HBM * B_L2 / (partial * B_L2 + (1 - partial) * B_HBM)
    return min(B_mix * I, Pm), I, B_mix

print(f'{"块(MxN)":>12} {"I (MAC/B)":>12} {"∂=1.0 P":>10} {"∂=0.5 P":>10} {"bound":>8}')
for bm, bn in [(16, 16), (64, 64), (128, 128), (256, 128), (256, 256)]:
    p1, I, _ = roofline(bm, bn, 1.0)     # 全 HBM
    p2, _, _ = roofline(bm, bn, 0.5)     # 一半命中 L2
    bound = '计算' if p1 >= Pm else '带宽'
    print(f'{f"{bm}x{bn}":>12} {I:>12.0f} {p1/1e12:>9.1f}T {p2/1e12:>9.1f}T {bound:>8}')

I_star = Pm / B_HBM
print(f'\n全 HBM 时拐点 I* = Pm/B_HBM = {I_star:.0f} MAC/B')
print('观察：块从 16x16 增大到 128x128，I 提升 8 倍；256x256 越过拐点进入计算受限平台；')
print('     命中率提高(∂ 下降)相当于抬高屋顶；越过拐点后再加大块只浪费 UB、不再涨性能。')